In [ ]:
# PyTorch functions/methods helpers

# 6.7.1
has_cuda = torch.cuda.is_available() # Checks whether CUDA is available for PyTorch to use
device = torch.device("cuda" if has_cuda else "cpu") # Creates a device object telling PyTorch where you want computations to happen
print("visible cuda devices:", torch.cuda.device_count()) # How many CUDA GPUs PyTorch can see (e.g. sometimes you want to place your model into other GPUs)

# 6.7.3
model = nn.Sequential(
    nn.Linear(3, 4), # This layer stays on the CPU (default) since it was not moved to "device"
    nn.ReLU(),
    nn.Linear(4, 1).to(device) # Moves registered parameters and buffers to the intended device
) # To move the whole thing to your intended device, you add .to(device) here
X = torch.randn(5, 3, device=device) # Setup which device do you wish to setup this param under
Y = model(X)

# 6.7.4
loss = (Y ** 2).mean() # Loss can be, and if you have cuda, is usually computed on cuda/Nvidia GPUs
loss_value = float(loss.detach().cpu()) # Move the loss to CPU only when needed for Python/logging; unnecessary GPU→CPU transfers can slow training

# 6.7.5
cpu_x = torch.randn(2, 3) # The user input randn(2, 3) is not ported into cuda (PyTorch default stays with CPU) like the gpu_model below, so gpu_model(cpu_x) will run into mismatch
gpu_model = nn.Linear(3, 1).to("cuda") # You can also write this as .cuda() or just .to(device) if you have it setup

* Deep learning computation happens on devices: usually CPU or GPU.

* Device management is the discipline of keeping tensors, parameters, buffers, losses, logging values etc. in the right place/time.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain why hardware placement is part of the tensor contract
- choose a CPU/GPU device safely
- move tensors and modules to the same device
- understand device mismatch errors
- move tensors back to CPU for logging or NumPy conversion

In [1]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_scalars(parameters):
    return sum(p.numel() for p in parameters)

# 6.7.0 The Problem This Notebook Solves

A tensor is not only a shape and dtype. It also lives somewhere.
* On a CPU-only machine, all computation happens on CPU.
* On a GPU machine, large tensor operations can be much faster if the relevant tensors and model parameters are on that GPU.
* PyTorch generally does not hide this from you because moving data across devices has real cost and can create ambiguity.

The core rule is:

```text
tensors that participate in one operation usually must live on the same device
```

This matters for model code:

- model parameters live on a device
- input batches live on a device
- losses and outputs live where they were computed
- logging often wants CPU scalar values

This notebook is CPU-safe. If a GPU exists, it uses it. If not, it still teaches the same placement logic.

# 6.7.1 Choose a Device Without Assuming a GPU Exists

Cloud notebooks vary. Some sessions have CUDA. Some do not. Good teaching code should not crash just because no GPU is available.

The pattern is:

```text
if CUDA is available, use cuda
otherwise, use cpu
```

This is not only convenience. It makes the rest of the notebook express a single device contract: every tensor and module should be moved to `device`, whatever that device happens to be.

In [2]:
has_cuda = torch.cuda.is_available() # Checks whether CUDA is available for PyTorch to use
device = torch.device("cuda" if has_cuda else "cpu") # Creates a device object telling PyTorch where you want computations to happen

print("cuda available:", has_cuda)
print("selected device:", device)
print("visible cuda devices:", torch.cuda.device_count()) # How many CUDA GPUs PyTorch can see (e.g. sometimes you want to place your model into other GPUs)

assert device.type in {"cpu", "cuda"}

cuda available: False
selected device: cpu
visible cuda devices: 0


# 6.7.2 Tensors Live on One Device at a Time

A tensor's device is part of its runtime identity.
* Two tensors with the same values and shapes are not directly compatible for most operations if one is on CPU and the other on GPU.
* This is because CPU and GPU memory are in different places. Adding tensors requires the operation to access both operands.
* PyTorch expects them to be colocated unless an operation explicitly handles transfer.

The cell creates both operands directly on the selected device, performs the addition there, then moves the result to CPU only for an equality check.

In [3]:
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = torch.ones(3, device=device)
z = x + y

print("x device:", x.device) # Check which device is x on
print("z:", z)

assert z.device == x.device
assert torch.equal(z.cpu(), torch.tensor([2.0, 3.0, 4.0]))

x device: cpu
z: tensor([2., 3., 4.])


# 6.7.3 Modules Move Through Their Parameters and Buffers

Calling `.to(device)` on a module moves registered parameters and buffers.

It does not create a permanent force field that pulls all future input tensors onto the same device.

That separation is important:

```text
model.to(device) moves model state
X.to(device) or tensor creation with device=device moves data
```

The forward pass succeeds only when model state and input batch are colocated.

The cell checks the set of parameter devices to prove that the module state moved.

In [4]:
model = nn.Sequential(
    nn.Linear(3, 4), # This layer stays on the CPU (default) since it was not moved to "device"
    nn.ReLU(),
    nn.Linear(4, 1).to(device) # Moves registered parameters and buffers to the intended device
) # To move the whole thing to your intended device, you add .to(device) here
X = torch.randn(5, 3, device=device) # Setup which device do you wish to setup this param under
Y = model(X)

parameter_devices = {p.device.type for p in model.parameters()}

print("parameter devices", parameter_devices)
print("output device:", Y.device)

assert parameter_devices == {device.type}
assert Y.device.type == device.type

parameter devices {'cpu'}
output device: cpu


# 6.7.4 Logging Usually Wants CPU Values

Training code often computes values on GPU but logs values in ordinary Python. That creates two separate issues:

- gradient history: the loss tensor is part of the computation graph
- device placement: the loss tensor may live on GPU
- `detach()` says "I want the value, not the graph history."
- `cpu()` says "move the value to CPU memory."
- Converting to `float` then gives a plain Python number that is safe for logging.

This distinction prevents accidental graph retention and avoids device-specific logging surprises.

In [5]:
loss = (Y ** 2).mean() # Loss can be, and if you have cuda, is usually computed on cuda/Nvidia GPUs
loss_value = float(loss.detach().cpu()) # Move the loss to CPU only when needed for Python/logging; unnecessary GPU→CPU transfers can slow training

print("loss tensor device:", loss.device)
print("loss Python loss:", loss_value)

assert isinstance(loss_value, float)

loss tensor device: cpu
loss Python loss: 0.021418122574687004


# 6.7.5 Break It Deliberately: Device Mismatch

Device mismatch errors are not mysterious. They usually mean one part of the computation is on CPU while another part is on GPU.

This cell creates that failure only when CUDA exists.

On CPU-only machines, it skips the failure because there is no GPU to mismatch against.

The conceptual mistake is splitting one forward computation across devices without an explicit transfer plan.

In [6]:
if torch.cuda.is_available():
  cpu_x = torch.randn(2, 3) # The user input randn(2, 3) is not ported into cuda (PyTorch default stays with CPU) like the gpu_model below, so gpu_model(cpu_x) will run into mismatch
  gpu_model = nn.Linear(3, 1).to("cuda") # You can also write this as .cuda() or just .to(device) if you have it setup
  try:
    gpu_model(cpu_x)
  except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
  else:
    raise AssertionError("A CPU tensor should not run through a CUDA model.")
else:
  print("Skipped mismatch demo because CUDA is not available in this runtime.")

Skipped mismatch demo because CUDA is not available in this runtime.


# 6.7.6 Minimal Device-Safe Training Step

This final cell is the device-safe pattern to carry forward:

```text
choose device
move model state to device
create or move batch tensors to device
compute predictions and loss on that device
backpropagate
update parameters
move small logging values back to CPU
```

The training logic is not different because a GPU exists. The difference is placement.

The same conceptual training loop from earlier chapters still applies; Chapter 6.7 adds the hardware contract.

In [7]:
# All of the .to(device) is intended to move to cuda if it exists, else to cpu

model = nn.Linear(3, 1).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

X = torch.randn(8, 3, device=device)
y = torch.randn(8, 1, device=device)

pred = model(X)
loss = ((pred - y) ** 2).mean()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("loss:", float(loss.detach().cpu())) # Copy loss value back to cpu and convert it to a Python float
print("model device:", next(model.parameters()).device) # Model device should be cuda if available, otherwise cpu

assert next(model.parameters()).device.type == device.type
assert torch.isfinite(loss).item() # Check that the loss is finite (not NaN or ±inf) to see if the training ran sucessfully on the intended device (cuda)

loss: 2.686006546020508
model device: cpu


# 6.7 Checkpoint

Answer these before moving on.

You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. Why is device part of a tensor's runtime contract?
> Because different device (cpu, cuda) will use different operands, and mixing those up during training (especially) will puke PyTorch out as it detects mismatch in devices

2. Why is device selection written with a CPU fallback?
> Because not all devices have cuda/Nvidia GPU. Matter of fact, you need to pay Google Colab to use a cuda instance! Whereas every computer/server needs a cpu

3. What does `model.to(device)` move, and what does it not move?
> It moves the model's registered parameters (e.g. user input X) and buffers to the specified device. It does not automatically move future input tensors/batches to that device (e.g. the model inside `nn.Sequential()` after initializing X)

4. Why does moving the model not automatically move future input batches?
> Keep the data/model involved in GPU computation on the GPU; keep unrelated work on CPU when appropriate

5. Why should logged scalar values often use `detach().cpu()`?
> Because logging usually only needs the numerical value, not its connection to the autograd computation graph. `detach()` removes the tensor from gradient tracking, and `.cpu()` moves the value to CPU so it can easily be converted to a Python scalar or used by CPU-side logging/plotting code

6. What does a device mismatch error mean mechanically?
> Means your model is not running on the same device (e.g. X or user input was in cpu but layer or model was in cuda)